<a href="https://colab.research.google.com/github/Elensk8/Anal-tica_Neg/blob/main/ProyectoFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Base de datos Churn (Banco)**

# **Equipo: Juan Carlos Cifuentes, Elena Restrepo y Juan Jose Puerta.**

**Implementamos el modelo de pronóstico de Redes Neuronales**

0. Cargar las librerías de trabajo

In [42]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

1. Instalamos los paquetes de trabajo

In [41]:
!pip install tensorflow

2. Cargar la base de datos y las variables que vamos a utilizar

In [93]:
nxl = '/content/Churn_Modelling.csv'
XDB = pd.read_csv(nxl)  # Usa read_csv si el archivo es .csv

# Selección de variables
XDB = XDB[['CustomerId', 'CreditScore', 'Age', 'Balance',
           'NumOfProducts', 'HasCrCard', 'IsActiveMember',
           'EstimatedSalary', 'Exited']]

# Variables de entrada (X) y salida (y)
XD = np.array(XDB[['CustomerId', 'CreditScore', 'Age', 'Balance',
                   'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                   'EstimatedSalary']])
XDn = XD / np.max(XD, axis=0)  # Normalización

yd = np.array(XDB['Exited'])  # Variable binaria (no se normaliza)

print("Los datos de entrada normalizados son:\n", XDn)

Los datos de entrada normalizados son:
 [[0.9885501  0.72823529 0.45652174 ... 1.         1.         0.50676345]
 [0.98935367 0.71529412 0.44565217 ... 0.         1.         0.56273406]
 [0.98758284 0.59058824 0.45652174 ... 1.         0.         0.56967927]
 ...
 [0.98538426 0.83411765 0.39130435 ... 0.         1.         0.21043581]
 [0.99156945 0.90823529 0.45652174 ... 1.         0.         0.46446006]
 [0.98815284 0.93176471 0.30434783 ... 1.         0.         0.19096108]]


##**MADALINE - MULTI ADALINE**
3. Se procede con el diseño del modelo MADALINE

**Con Softmax**

In [119]:
# Variables de entrada (X) y salida (y)
XD = np.array(XDB[[ 'CreditScore', 'Age', 'Balance',
                   'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                   'EstimatedSalary']])
XDn = XD / np.max(XD, axis=0)  # Normalización

yd = np.array(XDB['Exited'])  # Variable binaria (no se normaliza)

# Normalize yd and assign to ydn. Using max as a scaling factor.
# Note: For binary classification with a neural network, using the raw 0/1 values
# and binary_crossentropy loss with a sigmoid output activation is more standard.
# However, based on the original code using 'mse' loss, scaling might be intended.
#ydn = yd / np.max(yd)
from tensorflow.keras.utils import to_categorical

ydn=to_categorical(yd)  #aseguro que la variable es categorica


print("Los datos de entrada normalizados son:\n", XDn)
print("Los datos de salida normalizados son:\n", ydn) # Add this print for verification

# ##**MADALINE - MULTI ADALINE**
# 7. Se procede con el diseño del modelo MADALINE
# %%
madaline=tf.keras.models.Sequential([
    tf.keras.layers.Dense(10,input_shape=(7,),activation='relu',use_bias=False), #Capa 1 _ 10 refresiones lineales
    tf.keras.layers.Dense(2,activation='softmax',use_bias=False)]) #Capa 2: Nucleo - Salida del modelo
madaline.compile(optimizer='adam',loss='categorical_crossentropy') #sgd: Solve Descend Gradient. mse= Mean Square Error -Quiero que sea cero.

history=madaline.fit(XDn,ydn,epochs=100) #Recorra la tabla de datos 100 veces
ydp=madaline.predict(XDn) #pronóstico al final del aprendizaje
ydp2 =ydp.argmax(axis=1)

from sklearn.metrics import classification_report, confusion_matrix
print("Matriz de confusión:")
print(confusion_matrix(yd, ydp2))

cm=np.array(confusion_matrix(yd,ydp2))

VN=cm[0,0]; FP=cm[0,1]; FN=cm[1,0]; VP=cm[1,1]
Ex=(VN+VP)/(VN+FP+FN+VP)
print("Exactitud: ",Ex)

TE=(FN+FP)/(VN+FP+FN+VP)
print("Tasa de error: ",TE)



df = pd.DataFrame(np.column_stack((ydn,ydp)))
print("La correlación de los datos es:\n",df.corr())

#Obtenemos las conexiones de la red - Como se dispersan las conexiones nerviosas
WC=madaline.get_weights() # Estas son las capas
# There are only two layers and no bias, so WC will have two elements
W=WC[0];C=WC[1] #Capa 0 - 4 a 10: Capa 1 - 10 a 1 Salida. Conexiones nerviosas (10 regresiones)
a=W@C #Producto entre las conexiones como Ws pasan entre los C
     #a: son los efectos indepencientes de las variables sobre la salida
    # Cual efecto es mas importantr que otro y cual tiene un efecto negativo o positivo sobre la salida.
df2=pd.DataFrame(a.T)
df2.columns=[[ 'CreditScore', 'Age', 'Balance',
                   'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                   'EstimatedSalary']]
display(df2) #Muestra el peso que tiene cada variable  sobre la salida


Los datos de entrada normalizados son:
 [[0.72823529 0.45652174 0.         ... 1.         1.         0.50676345]
 [0.71529412 0.44565217 0.33403148 ... 0.         1.         0.56273406]
 [0.59058824 0.45652174 0.63635718 ... 1.         0.         0.56967927]
 ...
 [0.83411765 0.39130435 0.         ... 0.         1.         0.21043581]
 [0.90823529 0.45652174 0.29922631 ... 1.         0.         0.46446006]
 [0.93176471 0.30434783 0.51870777 ... 1.         0.         0.19096108]]
Los datos de salida normalizados son:
 [[0. 1.]
 [1. 0.]
 [0. 1.]
 ...
 [0. 1.]
 [0. 1.]
 [1. 0.]]
Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6322
Epoch 2/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4992
Epoch 3/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4880
Epoch 4/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4770
Epoch 5/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4714
Epoch 6/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4563
Epoch 7/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4681
Epoch 8/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4474
Epoch 9/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4453
Epoch 10/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4401
Epoch 11/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4314
Epoch 12/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4343
Epoch 13/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4332
Epoch 14/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4297
Epoch 15/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step -

,CreditScore,Age,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,5.585486,1.539018,2.258316,-8.443069,-0.207749,-0.417367,0.085908
1,-6.385105,1.540881,-2.372226,5.572896,-0.379017,0.821612,-0.379007


**Con Sigmoid**

In [120]:
# Variables de entrada (X) y salida (y)
XD = np.array(XDB[[ 'CreditScore', 'Age', 'Balance',
                   'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                   'EstimatedSalary']])
XDn = XD / np.max(XD, axis=0)  # Normalización

yd = np.array(XDB['Exited'])  # Variable binaria (no se normaliza)

# Normalize yd and assign to ydn. Using max as a scaling factor.
# Note: For binary classification with a neural network, using the raw 0/1 values
# and binary_crossentropy loss with a sigmoid output activation is more standard.
# However, based on the original code using 'mse' loss, scaling might be intended.
#ydn = yd / np.max(yd)
from tensorflow.keras.utils import to_categorical

ydn=to_categorical(yd)  #aseguro que la variable es categorica


print("Los datos de entrada normalizados son:\n", XDn)
print("Los datos de salida normalizados son:\n", ydn) # Add this print for verification

# ##**MADALINE - MULTI ADALINE**
# 7. Se procede con el diseño del modelo MADALINE
# %%
madaline=tf.keras.models.Sequential([
    tf.keras.layers.Dense(10,input_shape=(7,),activation='relu',use_bias=False), #Capa 1 _ 10 refresiones lineales
    tf.keras.layers.Dense(2,activation='sigmoid',use_bias=False)]) #Capa 2: Nucleo - Salida del modelo
madaline.compile(optimizer='adam',loss='categorical_crossentropy') #sgd: Solve Descend Gradient. mse= Mean Square Error -Quiero que sea cero.

history=madaline.fit(XDn,ydn,epochs=100) #Recorra la tabla de datos 100 veces
ydp=madaline.predict(XDn) #pronóstico al final del aprendizaje
ydp2 =ydp.argmax(axis=1)

from sklearn.metrics import classification_report, confusion_matrix
print("Matriz de confusión:")
print(confusion_matrix(yd, ydp2))

cm=np.array(confusion_matrix(yd,ydp2))

VN=cm[0,0]; FP=cm[0,1]; FN=cm[1,0]; VP=cm[1,1]
Ex=(VN+VP)/(VN+FP+FN+VP)
print("Exactitud: ",Ex)

TE=(FN+FP)/(VN+FP+FN+VP)
print("Tasa de error: ",TE)



df = pd.DataFrame(np.column_stack((ydn,ydp)))
print("La correlación de los datos es:\n",df.corr())

#Obtenemos las conexiones de la red - Como se dispersan las conexiones nerviosas
WC=madaline.get_weights() # Estas son las capas
# There are only two layers and no bias, so WC will have two elements
W=WC[0];C=WC[1] #Capa 0 - 4 a 10: Capa 1 - 10 a 1 Salida. Conexiones nerviosas (10 regresiones)
a=W@C #Producto entre las conexiones como Ws pasan entre los C
     #a: son los efectos indepencientes de las variables sobre la salida
    # Cual efecto es mas importantr que otro y cual tiene un efecto negativo o positivo sobre la salida.
df2=pd.DataFrame(a.T)
df2.columns=[[ 'CreditScore', 'Age', 'Balance',
                   'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                   'EstimatedSalary']]
display(df2) #Muestra el peso que tiene cada variable  sobre la salida

Los datos de entrada normalizados son:
 [[0.72823529 0.45652174 0.         ... 1.         1.         0.50676345]
 [0.71529412 0.44565217 0.33403148 ... 0.         1.         0.56273406]
 [0.59058824 0.45652174 0.63635718 ... 1.         0.         0.56967927]
 ...
 [0.83411765 0.39130435 0.         ... 0.         1.         0.21043581]
 [0.90823529 0.45652174 0.29922631 ... 1.         0.         0.46446006]
 [0.93176471 0.30434783 0.51870777 ... 1.         0.         0.19096108]]
Los datos de salida normalizados son:
 [[0. 1.]
 [1. 0.]
 [0. 1.]
 ...
 [0. 1.]
 [0. 1.]
 [1. 0.]]


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.6133
Epoch 2/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5169
Epoch 3/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.4935
Epoch 4/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4908
Epoch 5/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4771
Epoch 6/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4674
Epoch 7/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4715
Epoch 8/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4645
Epoch 9/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4455
Epoch 10/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4560
Epoch 11/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4427
Epoch 12/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4454
Epoch 13/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4367
Epoch 14/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4426
Epoch 15/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 1

,CreditScore,Age,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,4.968409,-6.006786,0.542667,-1.991011,-0.325108,-0.181211,0.558768
1,-6.034883,5.661724,0.556614,4.225431,-0.921632,-0.262222,-0.148000


**Implementamos modelo de agrupamiento**

4. Se procede con la implementación del modelo

*Max Depth es el número de variables disponibles

*Gini es el criterio que indica el grado de error en el modelo

In [3]:
import numpy as np
import pandas as pd

#Para implementación del modelo
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

**Árbol de decisión**

In [98]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz
# Ensure pydotplus is installed if you want to visualize the tree
!pip install pydotplus graphviz
from pydotplus import graph_from_dot_data
import pandas as pd # Make sure pandas is imported

# Reload the Churn_Modelling.csv data into a DataFrame
# This is necessary because XD and yd are now numpy arrays from the previous cell
nxl = '/content/Churn_Modelling.csv'
XDB_churn = pd.read_csv(nxl)

# Select the features (X) and the target (y)
# Exclude CustomerId as it's usually not a predictive feature
# Exclude Exited as it is the target variable
X_churn = XDB_churn[['CreditScore', 'Age', 'Balance',
                     'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                     'EstimatedSalary']]
y_churn = XDB_churn['Exited'] # The target variable is already 0/1, no need to encode

# If there were categorical features in X_churn, you would encode them here
# For example, if 'Gender' was in X_churn:
# X_churn = pd.get_dummies(X_churn, columns=['Gender'], drop_first=True)

# Árbol de Clasificación (since Exited is binary)
# Using DecisionTreeClassifier is appropriate for binary classification
mar = DecisionTreeClassifier(max_depth=4)

# Fit the classifier
mar.fit(X_churn, y_churn) # Fit using the DataFrame X_churn and Series y_churn

# Visualización
# Get the feature names from the processed feature DataFrame
vs = X_churn.columns.tolist()

# Generate the dot data for the tree visualization
dot_data = export_graphviz(mar, feature_names=vs, filled=True,
                           class_names=['Not Exited', 'Exited']) # Add class names for clarity

# Create the graph from the dot data
graph = graph_from_dot_data(dot_data)

# Save the tree visualization to a PNG file
graph.write_png('Arbol_Exited.png')

print("Decision Tree visualization saved as Arbol_Exited.png")

Decision Tree visualization saved as Arbol_Exited.png
